# Semantic Lossless — account A

Run the setup cell, configure native dataset/checkpoint paths, then run three cells per model. Pending integrations are skipped explicitly. GPU pilots must pass before training. No cell deletes checkpoints or earlier results.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json
# Mount a checkout containing the new src/semantic_lossless package.
# The historical remote commit alone does not contain this implementation yet.
REPO = Path('/kaggle/working/Semantic_Lossless')
assert (REPO / 'src/semantic_lossless/cli.py').is_file(), 'Mount/clone the updated implementation checkout first'
os.chdir(REPO)
os.environ['SEMANTIC_LOSSLESS_ROOT'] = str(REPO)
sys.path.insert(0,str(REPO/'src'))
from semantic_lossless.catalog import get_model
# Set only native paths here. Credentials never belong in notebooks.
CONFIGS = {}  # e.g. {'jcm': '/kaggle/working/jcm.json'}
TRAIN_MODELS = set()  # Explicitly select audited native recipes; generic examples are adapted runs.
READY = {}
TRAINED = {}
def command(stage, config, inference_only=False):
    extra = ['--inference-only'] if inference_only else []
    subprocess.run([sys.executable,'-u','semcom.py',stage,'--config',str(config)]+extra,check=True)
def latest_run(config):
    data=json.loads(Path(config).read_text())
    root=Path(data.get('output_root','Results/experiments'))/data['algorithm']
    return max(root.iterdir(),key=lambda p:p.stat().st_mtime_ns)


## deepjscc
Prepare/pilot → train or retain → evaluate/profile/export.

In [ ]:
name = 'deepjscc'
config = CONFIGS.get(name)
spec = get_model(name)
READY[name] = False
if not config:
    print(name, 'SKIPPED: configure its native dataset and checkpoint paths')
elif spec.backend in ('pending','unavailable'):
    print(name, 'BLOCKED:',spec.note)
else:
    command('prepare',config)
    if name == 'shared_semantic_codebook':
        data=json.loads(Path(config).read_text())
        if not data.get('model_options', {}).get('codebook'):
            command('construct',config)
            construction_run=latest_run(config)
            book=construction_run/'codebook.pt'
            if not book.is_file():
                raise RuntimeError('Codebook construction did not complete')
            data.setdefault('model_options', {})['codebook']=str(book.resolve())
            prepared_config=construction_run/'codec_config.json'
            prepared_config.write_text(json.dumps(data,indent=2))
            CONFIGS[name]=config=str(prepared_config.resolve())
    command('pilot',config,inference_only=name not in TRAIN_MODELS)
    READY[name] = True


In [ ]:
name = 'deepjscc'
if READY.get(name) and get_model(name).trainable and name in TRAIN_MODELS:
    command('train',CONFIGS[name])
    TRAINED[name] = latest_run(CONFIGS[name])
    # Evaluate the newly trained best checkpoint rather than original weights.
    best = TRAINED[name]/'best.pt'
    if not best.is_file():
        raise RuntimeError('No fully validated best checkpoint; inspect run status before evaluation')
    data=json.loads(Path(CONFIGS[name]).read_text())
    data['checkpoint']=str(best.resolve())
    data.setdefault('model_options', {})['from_scratch']=False
    evaluation_config=TRAINED[name]/'evaluation_config.json'
    evaluation_config.write_text(json.dumps(data,indent=2))
    CONFIGS[name]=str(evaluation_config.resolve())
elif READY.get(name):
    print(name, 'Retaining pretrained weights; long training not selected in this launcher')
else:
    print(name, 'SKIPPED: pilot has not passed')


In [ ]:
name = 'deepjscc'
if READY.get(name):
    command('evaluate',CONFIGS[name])
    evaluation_run=latest_run(CONFIGS[name])
    command('profile',CONFIGS[name])
    subprocess.run([sys.executable,'semcom.py','export','--run-dir',str(evaluation_run)],check=True)
    subprocess.run([sys.executable,'semcom.py','collect'],check=True)
    # Kaggle retains /kaggle/working as notebook output after Save & Run All.
    # Results remain in algorithm/run folders; collection merges them by event ID.
else:
    print(name, 'SKIPPED: no verified pilot')


## ntscc
Prepare/pilot → train or retain → evaluate/profile/export.

In [ ]:
name = 'ntscc'
config = CONFIGS.get(name)
spec = get_model(name)
READY[name] = False
if not config:
    print(name, 'SKIPPED: configure its native dataset and checkpoint paths')
elif spec.backend in ('pending','unavailable'):
    print(name, 'BLOCKED:',spec.note)
else:
    command('prepare',config)
    if name == 'shared_semantic_codebook':
        data=json.loads(Path(config).read_text())
        if not data.get('model_options', {}).get('codebook'):
            command('construct',config)
            construction_run=latest_run(config)
            book=construction_run/'codebook.pt'
            if not book.is_file():
                raise RuntimeError('Codebook construction did not complete')
            data.setdefault('model_options', {})['codebook']=str(book.resolve())
            prepared_config=construction_run/'codec_config.json'
            prepared_config.write_text(json.dumps(data,indent=2))
            CONFIGS[name]=config=str(prepared_config.resolve())
    command('pilot',config,inference_only=name not in TRAIN_MODELS)
    READY[name] = True


In [ ]:
name = 'ntscc'
if READY.get(name) and get_model(name).trainable and name in TRAIN_MODELS:
    command('train',CONFIGS[name])
    TRAINED[name] = latest_run(CONFIGS[name])
    # Evaluate the newly trained best checkpoint rather than original weights.
    best = TRAINED[name]/'best.pt'
    if not best.is_file():
        raise RuntimeError('No fully validated best checkpoint; inspect run status before evaluation')
    data=json.loads(Path(CONFIGS[name]).read_text())
    data['checkpoint']=str(best.resolve())
    data.setdefault('model_options', {})['from_scratch']=False
    evaluation_config=TRAINED[name]/'evaluation_config.json'
    evaluation_config.write_text(json.dumps(data,indent=2))
    CONFIGS[name]=str(evaluation_config.resolve())
elif READY.get(name):
    print(name, 'Retaining pretrained weights; long training not selected in this launcher')
else:
    print(name, 'SKIPPED: pilot has not passed')


In [ ]:
name = 'ntscc'
if READY.get(name):
    command('evaluate',CONFIGS[name])
    evaluation_run=latest_run(CONFIGS[name])
    command('profile',CONFIGS[name])
    subprocess.run([sys.executable,'semcom.py','export','--run-dir',str(evaluation_run)],check=True)
    subprocess.run([sys.executable,'semcom.py','collect'],check=True)
    # Kaggle retains /kaggle/working as notebook output after Save & Run All.
    # Results remain in algorithm/run folders; collection merges them by event ID.
else:
    print(name, 'SKIPPED: no verified pilot')


## sqgan
Prepare/pilot → train or retain → evaluate/profile/export.

In [ ]:
name = 'sqgan'
config = CONFIGS.get(name)
spec = get_model(name)
READY[name] = False
if not config:
    print(name, 'SKIPPED: configure its native dataset and checkpoint paths')
elif spec.backend in ('pending','unavailable'):
    print(name, 'BLOCKED:',spec.note)
else:
    command('prepare',config)
    if name == 'shared_semantic_codebook':
        data=json.loads(Path(config).read_text())
        if not data.get('model_options', {}).get('codebook'):
            command('construct',config)
            construction_run=latest_run(config)
            book=construction_run/'codebook.pt'
            if not book.is_file():
                raise RuntimeError('Codebook construction did not complete')
            data.setdefault('model_options', {})['codebook']=str(book.resolve())
            prepared_config=construction_run/'codec_config.json'
            prepared_config.write_text(json.dumps(data,indent=2))
            CONFIGS[name]=config=str(prepared_config.resolve())
    command('pilot',config,inference_only=name not in TRAIN_MODELS)
    READY[name] = True


In [ ]:
name = 'sqgan'
if READY.get(name) and get_model(name).trainable and name in TRAIN_MODELS:
    command('train',CONFIGS[name])
    TRAINED[name] = latest_run(CONFIGS[name])
    # Evaluate the newly trained best checkpoint rather than original weights.
    best = TRAINED[name]/'best.pt'
    if not best.is_file():
        raise RuntimeError('No fully validated best checkpoint; inspect run status before evaluation')
    data=json.loads(Path(CONFIGS[name]).read_text())
    data['checkpoint']=str(best.resolve())
    data.setdefault('model_options', {})['from_scratch']=False
    evaluation_config=TRAINED[name]/'evaluation_config.json'
    evaluation_config.write_text(json.dumps(data,indent=2))
    CONFIGS[name]=str(evaluation_config.resolve())
elif READY.get(name):
    print(name, 'Retaining pretrained weights; long training not selected in this launcher')
else:
    print(name, 'SKIPPED: pilot has not passed')


In [ ]:
name = 'sqgan'
if READY.get(name):
    command('evaluate',CONFIGS[name])
    evaluation_run=latest_run(CONFIGS[name])
    command('profile',CONFIGS[name])
    subprocess.run([sys.executable,'semcom.py','export','--run-dir',str(evaluation_run)],check=True)
    subprocess.run([sys.executable,'semcom.py','collect'],check=True)
    # Kaggle retains /kaggle/working as notebook output after Save & Run All.
    # Results remain in algorithm/run folders; collection merges them by event ID.
else:
    print(name, 'SKIPPED: no verified pilot')


## adjscc
Prepare/pilot → train or retain → evaluate/profile/export.

In [ ]:
name = 'adjscc'
config = CONFIGS.get(name)
spec = get_model(name)
READY[name] = False
if not config:
    print(name, 'SKIPPED: configure its native dataset and checkpoint paths')
elif spec.backend in ('pending','unavailable'):
    print(name, 'BLOCKED:',spec.note)
else:
    command('prepare',config)
    if name == 'shared_semantic_codebook':
        data=json.loads(Path(config).read_text())
        if not data.get('model_options', {}).get('codebook'):
            command('construct',config)
            construction_run=latest_run(config)
            book=construction_run/'codebook.pt'
            if not book.is_file():
                raise RuntimeError('Codebook construction did not complete')
            data.setdefault('model_options', {})['codebook']=str(book.resolve())
            prepared_config=construction_run/'codec_config.json'
            prepared_config.write_text(json.dumps(data,indent=2))
            CONFIGS[name]=config=str(prepared_config.resolve())
    command('pilot',config,inference_only=name not in TRAIN_MODELS)
    READY[name] = True


In [ ]:
name = 'adjscc'
if READY.get(name) and get_model(name).trainable and name in TRAIN_MODELS:
    command('train',CONFIGS[name])
    TRAINED[name] = latest_run(CONFIGS[name])
    # Evaluate the newly trained best checkpoint rather than original weights.
    best = TRAINED[name]/'best.pt'
    if not best.is_file():
        raise RuntimeError('No fully validated best checkpoint; inspect run status before evaluation')
    data=json.loads(Path(CONFIGS[name]).read_text())
    data['checkpoint']=str(best.resolve())
    data.setdefault('model_options', {})['from_scratch']=False
    evaluation_config=TRAINED[name]/'evaluation_config.json'
    evaluation_config.write_text(json.dumps(data,indent=2))
    CONFIGS[name]=str(evaluation_config.resolve())
elif READY.get(name):
    print(name, 'Retaining pretrained weights; long training not selected in this launcher')
else:
    print(name, 'SKIPPED: pilot has not passed')


In [ ]:
name = 'adjscc'
if READY.get(name):
    command('evaluate',CONFIGS[name])
    evaluation_run=latest_run(CONFIGS[name])
    command('profile',CONFIGS[name])
    subprocess.run([sys.executable,'semcom.py','export','--run-dir',str(evaluation_run)],check=True)
    subprocess.run([sys.executable,'semcom.py','collect'],check=True)
    # Kaggle retains /kaggle/working as notebook output after Save & Run All.
    # Results remain in algorithm/run folders; collection merges them by event ID.
else:
    print(name, 'SKIPPED: no verified pilot')


## shared_semantic_codebook
Prepare/pilot → train or retain → evaluate/profile/export.

In [ ]:
name = 'shared_semantic_codebook'
config = CONFIGS.get(name)
spec = get_model(name)
READY[name] = False
if not config:
    print(name, 'SKIPPED: configure its native dataset and checkpoint paths')
elif spec.backend in ('pending','unavailable'):
    print(name, 'BLOCKED:',spec.note)
else:
    command('prepare',config)
    if name == 'shared_semantic_codebook':
        data=json.loads(Path(config).read_text())
        if not data.get('model_options', {}).get('codebook'):
            command('construct',config)
            construction_run=latest_run(config)
            book=construction_run/'codebook.pt'
            if not book.is_file():
                raise RuntimeError('Codebook construction did not complete')
            data.setdefault('model_options', {})['codebook']=str(book.resolve())
            prepared_config=construction_run/'codec_config.json'
            prepared_config.write_text(json.dumps(data,indent=2))
            CONFIGS[name]=config=str(prepared_config.resolve())
    command('pilot',config,inference_only=name not in TRAIN_MODELS)
    READY[name] = True


In [ ]:
name = 'shared_semantic_codebook'
if READY.get(name) and get_model(name).trainable and name in TRAIN_MODELS:
    command('train',CONFIGS[name])
    TRAINED[name] = latest_run(CONFIGS[name])
    # Evaluate the newly trained best checkpoint rather than original weights.
    best = TRAINED[name]/'best.pt'
    if not best.is_file():
        raise RuntimeError('No fully validated best checkpoint; inspect run status before evaluation')
    data=json.loads(Path(CONFIGS[name]).read_text())
    data['checkpoint']=str(best.resolve())
    data.setdefault('model_options', {})['from_scratch']=False
    evaluation_config=TRAINED[name]/'evaluation_config.json'
    evaluation_config.write_text(json.dumps(data,indent=2))
    CONFIGS[name]=str(evaluation_config.resolve())
elif READY.get(name):
    print(name, 'Retaining pretrained weights; long training not selected in this launcher')
else:
    print(name, 'SKIPPED: pilot has not passed')


In [ ]:
name = 'shared_semantic_codebook'
if READY.get(name):
    command('evaluate',CONFIGS[name])
    evaluation_run=latest_run(CONFIGS[name])
    command('profile',CONFIGS[name])
    subprocess.run([sys.executable,'semcom.py','export','--run-dir',str(evaluation_run)],check=True)
    subprocess.run([sys.executable,'semcom.py','collect'],check=True)
    # Kaggle retains /kaggle/working as notebook output after Save & Run All.
    # Results remain in algorithm/run folders; collection merges them by event ID.
else:
    print(name, 'SKIPPED: no verified pilot')


## diffcom
Prepare/pilot → train or retain → evaluate/profile/export.

In [ ]:
name = 'diffcom'
config = CONFIGS.get(name)
spec = get_model(name)
READY[name] = False
if not config:
    print(name, 'SKIPPED: configure its native dataset and checkpoint paths')
elif spec.backend in ('pending','unavailable'):
    print(name, 'BLOCKED:',spec.note)
else:
    command('prepare',config)
    if name == 'shared_semantic_codebook':
        data=json.loads(Path(config).read_text())
        if not data.get('model_options', {}).get('codebook'):
            command('construct',config)
            construction_run=latest_run(config)
            book=construction_run/'codebook.pt'
            if not book.is_file():
                raise RuntimeError('Codebook construction did not complete')
            data.setdefault('model_options', {})['codebook']=str(book.resolve())
            prepared_config=construction_run/'codec_config.json'
            prepared_config.write_text(json.dumps(data,indent=2))
            CONFIGS[name]=config=str(prepared_config.resolve())
    command('pilot',config,inference_only=name not in TRAIN_MODELS)
    READY[name] = True


In [ ]:
name = 'diffcom'
if READY.get(name) and get_model(name).trainable and name in TRAIN_MODELS:
    command('train',CONFIGS[name])
    TRAINED[name] = latest_run(CONFIGS[name])
    # Evaluate the newly trained best checkpoint rather than original weights.
    best = TRAINED[name]/'best.pt'
    if not best.is_file():
        raise RuntimeError('No fully validated best checkpoint; inspect run status before evaluation')
    data=json.loads(Path(CONFIGS[name]).read_text())
    data['checkpoint']=str(best.resolve())
    data.setdefault('model_options', {})['from_scratch']=False
    evaluation_config=TRAINED[name]/'evaluation_config.json'
    evaluation_config.write_text(json.dumps(data,indent=2))
    CONFIGS[name]=str(evaluation_config.resolve())
elif READY.get(name):
    print(name, 'Retaining pretrained weights; long training not selected in this launcher')
else:
    print(name, 'SKIPPED: pilot has not passed')


In [ ]:
name = 'diffcom'
if READY.get(name):
    command('evaluate',CONFIGS[name])
    evaluation_run=latest_run(CONFIGS[name])
    command('profile',CONFIGS[name])
    subprocess.run([sys.executable,'semcom.py','export','--run-dir',str(evaluation_run)],check=True)
    subprocess.run([sys.executable,'semcom.py','collect'],check=True)
    # Kaggle retains /kaggle/working as notebook output after Save & Run All.
    # Results remain in algorithm/run folders; collection merges them by event ID.
else:
    print(name, 'SKIPPED: no verified pilot')


## hybrid_semantic_complementary
Prepare/pilot → train or retain → evaluate/profile/export.

In [ ]:
name = 'hybrid_semantic_complementary'
config = CONFIGS.get(name)
spec = get_model(name)
READY[name] = False
if not config:
    print(name, 'SKIPPED: configure its native dataset and checkpoint paths')
elif spec.backend in ('pending','unavailable'):
    print(name, 'BLOCKED:',spec.note)
else:
    command('prepare',config)
    if name == 'shared_semantic_codebook':
        data=json.loads(Path(config).read_text())
        if not data.get('model_options', {}).get('codebook'):
            command('construct',config)
            construction_run=latest_run(config)
            book=construction_run/'codebook.pt'
            if not book.is_file():
                raise RuntimeError('Codebook construction did not complete')
            data.setdefault('model_options', {})['codebook']=str(book.resolve())
            prepared_config=construction_run/'codec_config.json'
            prepared_config.write_text(json.dumps(data,indent=2))
            CONFIGS[name]=config=str(prepared_config.resolve())
    command('pilot',config,inference_only=name not in TRAIN_MODELS)
    READY[name] = True


In [ ]:
name = 'hybrid_semantic_complementary'
if READY.get(name) and get_model(name).trainable and name in TRAIN_MODELS:
    command('train',CONFIGS[name])
    TRAINED[name] = latest_run(CONFIGS[name])
    # Evaluate the newly trained best checkpoint rather than original weights.
    best = TRAINED[name]/'best.pt'
    if not best.is_file():
        raise RuntimeError('No fully validated best checkpoint; inspect run status before evaluation')
    data=json.loads(Path(CONFIGS[name]).read_text())
    data['checkpoint']=str(best.resolve())
    data.setdefault('model_options', {})['from_scratch']=False
    evaluation_config=TRAINED[name]/'evaluation_config.json'
    evaluation_config.write_text(json.dumps(data,indent=2))
    CONFIGS[name]=str(evaluation_config.resolve())
elif READY.get(name):
    print(name, 'Retaining pretrained weights; long training not selected in this launcher')
else:
    print(name, 'SKIPPED: pilot has not passed')


In [ ]:
name = 'hybrid_semantic_complementary'
if READY.get(name):
    command('evaluate',CONFIGS[name])
    evaluation_run=latest_run(CONFIGS[name])
    command('profile',CONFIGS[name])
    subprocess.run([sys.executable,'semcom.py','export','--run-dir',str(evaluation_run)],check=True)
    subprocess.run([sys.executable,'semcom.py','collect'],check=True)
    # Kaggle retains /kaggle/working as notebook output after Save & Run All.
    # Results remain in algorithm/run folders; collection merges them by event ID.
else:
    print(name, 'SKIPPED: no verified pilot')
